# Face Recognition: iResNet50 + Angular Margin Loss Comparison
**ArcFace Paper Configuration (Deng et al., 2019)**

## Experiments Pipeline
1. Project Root & GPU Init
2. Main Training: ArcFace + iResNet50
3. Loss Function Comparison (ArcFace vs CosFace vs SphereFace vs Softmax)
4. Verification Evaluation (EER, TAR@FAR, ROC)
5. Embedding Visualization (t-SNE)

**Results are saved in `results/{loss_type}/` subfolders.**
**Resume:** `--resume` now restores the latest full checkpoint: model weights, loss head, optimizer state, and next epoch.


---
## 1. Project Root & GPU Init

In [1]:
import os
import sys
import tensorflow as tf

# Change directory to project root so relative paths work properly
# It looks for 'dataset_final' to ensure we are in the correct root
while not os.path.exists('dataset_final') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)
print(f'Working directory changed to: {os.getcwd()}')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s) Detected: {len(gpus)}')
else:
    print('No GPU detected - running on CPU.')

Working directory changed to: c:\Users\Admin\Desktop\Project_DAT301m
GPU(s) Detected: 1


---
## 2. Main Training: ArcFace + iResNet50

**Config:** SGD(lr=0.01, momentum=0.9), Step Decay [16,22,26,30], s=64, margin 0→0.5 step-wise

Results → `results/arcface/`
**Resume:** `--resume` now restores the latest full checkpoint: model weights, loss head, optimizer state, and next epoch.


In [3]:
import importlib
from models.face_recognition.iresnet import train
importlib.reload(train)
from models.face_recognition.iresnet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'ms1m_arcface_dataset/train',
    '--val_dir', 'ms1m_arcface_dataset/val',
    '--test_dir', 'ms1m_arcface_dataset/test',
    '--loss_type', 'arcface',
    '--epochs', '80',
    '--batch_size', '128',
    '--lr', '0.01',
    '--dropout', '0.5',
    '--arcface_scale', '64.0',
    '--patience', '15',
    '--verify_every', '3',
    '--verify_pairs', '5000',
    '--resume',
]

train_main()

[GPU] Memory Growth enabled. 1 GPU(s) detected.
[Data] Loaded 4748639 train images, 491516 val images across 76504 classes.
[Resume Warning] No checkpoint found. Starting from scratch.
Model: "IRESNET50_Backbone"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 112, 112, 3  0           []                               
                                )]                                                                
                                                                                                  
 stem_conv (Conv2D)             (None, 112, 112, 64  1728        ['image_input[0][0]']            
                                )                                                                 
                                                                                              

KeyboardInterrupt: 

---
## 3. Loss Function Comparison

Each loss saves to its own subfolder: `results/cosface/`, `results/sphereface/`, `results/softmax/`

In [ ]:
# Train with CosFace
import importlib
from models.face_recognition.iresnet import train
importlib.reload(train)
from models.face_recognition.iresnet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'dataset_final/train',
    '--val_dir', 'dataset_final/val',
    '--test_dir', 'dataset_final/test',
    '--loss_type', 'cosface',
    '--epochs', '30',
    '--batch_size', '128',
    '--lr', '0.01',
    '--dropout', '0.4',
    '--arcface_scale', '64.0',
    '--verify_every', '3',
    '--verify_pairs', '25000',
    '--resume',
]

train_main()

In [ ]:
# Train with SphereFace
import importlib
from models.face_recognition.iresnet import train
importlib.reload(train)
from models.face_recognition.iresnet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'dataset_final/train',
    '--val_dir', 'dataset_final/val',
    '--test_dir', 'dataset_final/test',
    '--loss_type', 'sphereface',
    '--epochs', '30',
    '--batch_size', '128',
    '--lr', '0.01',
    '--dropout', '0.4',
    '--arcface_scale', '64.0',
    '--verify_every', '3',
    '--verify_pairs', '25000',
    '--resume',
]

train_main()

In [ ]:
# Train with Softmax (baseline — no margin)
import importlib
from models.face_recognition.iresnet import train
importlib.reload(train)
from models.face_recognition.iresnet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'dataset_final/train',
    '--val_dir', 'dataset_final/val',
    '--test_dir', 'dataset_final/test',
    '--loss_type', 'softmax',
    '--epochs', '30',
    '--batch_size', '128',
    '--lr', '0.01',
    '--dropout', '0.4',
    '--arcface_scale', '64.0',
    '--verify_every', '3',
    '--verify_pairs', '25000',
    '--resume',
]

train_main()

---
## 4. Verification Evaluation

Evaluate each loss type's backbone on unseen test identities.

**Change `--weights` and `--output_dir` to match the loss type you want to evaluate.**

In [2]:
# Evaluate ArcFace backbone
import importlib
from shared import eval_verification
importlib.reload(eval_verification)
from shared.eval_verification import main as eval_main

sys.argv = [
    'eval_verification.py',
    '--test_dir', 'dataset_final/test',
    '--weights', 'models/face_recognition/iresnet/results/arcface/best_iresnet50_backbone.h5',
    '--variant', 'iresnet50',
    '--embedding_dim', '512',
    '--pairs', '25000',
    '--output_dir', 'models/face_recognition/iresnet/results/arcface',
]

eval_main()


Face Verification Evaluation
  Variant:    iresnet50
  Weights:    models/face_recognition/iresnet/results/arcface/best_iresnet50_backbone.h5
  Test Dir:   dataset_final/test
  Identities: 108
  Images:     3240
  Pairs:      25000 pos + 25000 neg

[1/3] Extracting embeddings...
[2/3] Building pairs...
[3/3] Computing metrics...

Verification Results (108 unseen identities)
  Identities:     108
  Images used:    3240
  Pairs:          25000 pos + 25000 neg
------------------------------------------------------------
  EER:            0.1254 (12.54%)
  EER Threshold:  0.1604
  AUC:            0.9450
  TAR @ FAR=1e-2: 0.5326 (53.26%)
  TAR @ FAR=1e-3: 0.2563 (25.63%)
  TAR @ FAR=1e-4: 0.0795 (7.95%)
------------------------------------------------------------
  Pos sim (mean): 0.3531 +/- 0.1648
  Neg sim (mean): 0.0142 +/- 0.1266
[Plot] ROC curve saved to: models/face_recognition/iresnet/results/arcface\roc_curve.png
[Plot] Score distribution saved to: models/face_recognition/iresnet/r

---
## 5. Embedding Visualization (t-SNE)

In [3]:
from shared.visualize_embeddings import main as viz_main

sys.argv = [
    'visualize_embeddings.py',
    '--test_dir', 'dataset_final/test',
    '--weights', 'models/face_recognition/iresnet/results/arcface/best_iresnet50_backbone.h5',
    '--variant', 'iresnet50',
    '--max_ids', '20',
    '--max_images_per_id', '15',
    '--method', 'both',
    '--output_dir', 'models/face_recognition/iresnet/results/arcface',
]

viz_main()

Visualizing 20 identities
Total images: 300
Extracting embeddings...
3/3 [==============================] - 2s 344ms/step

Running t-SNE...
[Plot] Saved: models\face_recognition\iresnet\results\arcface\tsne_embeddings.png

Running PCA...
[Plot] Saved: models\face_recognition\iresnet\results\arcface\pca_embeddings.png

Computing class centroids...
[Plot] Saved: models\face_recognition\iresnet\results\arcface\similarity_heatmap.png

Class Centroid Statistics:
  Intra-class sim: 1.0000 (should be ~1.0)
  Inter-class sim: 0.0833 +/- 0.3327 (should be low)
  Gap:             0.9167


---
## 6. Two-Image Similarity Test

In [11]:
import numpy as np
from PIL import Image
from models.face_recognition.iresnet.iresnet50 import iResNet_Backbone

# ── Configuration ──
WEIGHTS = 'models/face_recognition/iresnet/results/arcface/best_iresnet50_backbone.h5'
IMG_SIZE = 112
THRESHOLD = 0.18  # Adjust based on EER threshold from evaluation

# ── Load backbone ──
backbone = iResNet_Backbone(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    embedding_dim=512,
    dropout_rate=0.4,
    normalize_embeddings=True,
)
backbone.load_weights(WEIGHTS)
print(f'Loaded weights from: {WEIGHTS}')

def preprocess(path):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.asarray(img, dtype=np.float32)
    return (arr - 127.5) / 128.0

# ── Compare two images ──
img_origin = r'C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\origin.jpg'   # ← Replace with actual path
img_sample = r'C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\test8.jpg'   # ← Replace with actual path

emb1 = backbone.predict(np.expand_dims(preprocess(img_origin), 0), verbose=0)[0]
emb2 = backbone.predict(np.expand_dims(preprocess(img_sample), 0), verbose=0)[0]

similarity = np.dot(emb1, emb2)
is_same = similarity > THRESHOLD

print(f'Cosine Similarity: {similarity:.4f}')
print(f'Threshold:         {THRESHOLD}')
print(f'Result:            {"SAME PERSON" if is_same else "DIFFERENT PERSON"}')

Loaded weights from: models/face_recognition/iresnet/results/arcface/best_iresnet50_backbone.h5
Cosine Similarity: 0.2381
Threshold:         0.18
Result:            SAME PERSON
